In [25]:
!pip install pyspark

In [26]:
!pip install kagglehub -q

In [27]:
import kagglehub
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
import math

In [28]:
# Создание сессии Spark
spark = SparkSession.builder \
    .appName("SF_Bike_Analysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("SparkSession создана!")
print(f"Версия Spark: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

SparkSession создана!
Версия Spark: 4.0.2
Spark UI: http://6e1d307ceb2d:4040


In [29]:
path = kagglehub.dataset_download("benhamner/sf-bay-area-bike-share")
print(f"Датасет скачан в: {path}")

print("\nФайлы в датасете:")
for file in os.listdir(path):
    print(f"  - {file}")

# загружаем файлы
trips_path = f"{path}/trip.csv"
stations_path = f"{path}/station.csv"

# читаем CSV
trips = spark.read.option("header", True).option("inferSchema", True).csv(trips_path)
stations = spark.read.option("header", True).option("inferSchema", True).csv(stations_path)

# преобразуем типы
trips = trips.withColumn("duration", F.col("duration").cast("int"))
stations = stations.withColumn("lat", F.col("lat").cast("double"))
stations = stations.withColumn("long", F.col("long").cast("double"))

print("\nПервые 3 поездки:")
trips.show(3)

print("\nСтанции:")
stations.show(5)

Using Colab cache for faster access to the 'sf-bay-area-bike-share' dataset.
Датасет скачан в: /kaggle/input/sf-bay-area-bike-share

Файлы в датасете:
  - status.csv
  - weather.csv
  - station.csv
  - trip.csv
  - database.sqlite

Первые 3 поездки:
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|     start_date|  start_station_name|start_station_id|       end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|8/29/2013 14:13|South Van Ness at...|              66|8/29/2013 14:14|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|      70|8/29/2013 14:42|  San Jose City Hall|              10|8/29/2013 14:43|  San Jose City Hall|            10| 

# Найти велосипед с максимальным временем пробега

In [30]:
top_bike = trips.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total_seconds")) \
    .withColumn("total_hours", F.col("total_seconds") / 3600) \
    .orderBy(F.desc("total_seconds"))

top_bike_id = top_bike.first()["bike_id"]
top_bike_hours = top_bike.first()["total_hours"]

print(f"Велосипед {top_bike_id} - {top_bike_hours:.2f} часов пробега")
top_bike.show(1)

Велосипед 535 - 5169.91 часов пробега
+-------+-------------+-----------------+
|bike_id|total_seconds|      total_hours|
+-------+-------------+-----------------+
|    535|     18611693|5169.914722222222|
+-------+-------------+-----------------+
only showing top 1 row


# Найти наибольшее геодезическое расстояние между станциями

In [31]:
st1 = stations.alias("st1")
st2 = stations.alias("st2")

distances = st1.crossJoin(st2) \
    .filter(F.col("st1.id") < F.col("st2.id")) \
    .withColumn("lat1", F.radians(F.col("st1.lat"))) \
    .withColumn("lat2", F.radians(F.col("st2.lat"))) \
    .withColumn("dLat", F.radians(F.col("st2.lat") - F.col("st1.lat"))) \
    .withColumn("dLon", F.radians(F.col("st2.long") - F.col("st1.long"))) \
    .withColumn(
        "distance_km",
        2 * 6371 * F.asin(
            F.sqrt(
                F.pow(F.sin(F.col("dLat") / 2), 2) +
                F.cos(F.col("lat1")) * F.cos(F.col("lat2")) *
                F.pow(F.sin(F.col("dLon") / 2), 2)
            )
        )
    ) \
    .select(
        F.col("st1.name").alias("station1_name"),
        F.col("st2.name").alias("station2_name"),
        F.col("distance_km")
    )

max_distance = distances.orderBy(F.desc("distance_km"))
print("Максимальное расстояние между станциями:")
max_distance.show(1, truncate=False)

Максимальное расстояние между станциями:
+--------------------------+----------------------+----------------+
|station1_name             |station2_name         |distance_km     |
+--------------------------+----------------------+----------------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|69.9208759542826|
+--------------------------+----------------------+----------------+
only showing top 1 row


# Найти путь велосипеда с максимальным временем пробега через станции

In [32]:
bike_path = trips.filter(F.col("bike_id") == top_bike_id) \
    .orderBy("start_date") \
    .select("start_station_name", "end_station_name", "duration")

print(f"Маршрут велосипеда {top_bike_id} (первые 20 поездок):")
bike_path.show(20, truncate=False)

print(f"\nВсего поездок у этого велосипеда: {bike_path.count()}")

Маршрут велосипеда 535 (первые 20 поездок):
+---------------------------------------------+---------------------------------------------+--------+
|start_station_name                           |end_station_name                             |duration|
+---------------------------------------------+---------------------------------------------+--------+
|Mechanics Plaza (Market at Battery)          |Embarcadero at Sansome                       |3289    |
|Embarcadero at Sansome                       |Market at 4th                                |1286    |
|Market at 4th                                |South Van Ness at Market                     |795     |
|Market at 10th                               |Powell Street BART                           |235     |
|Embarcadero at Folsom                        |San Francisco Caltrain (Townsend at 4th)     |596     |
|San Francisco Caltrain (Townsend at 4th)     |Temporary Transbay Terminal (Howard at Beale)|600     |
|Temporary Transbay Terminal 

# Найти количество велосипедов в системе

In [33]:
unique_bikes = trips.select("bike_id").distinct().count()
print(f"Всего уникальных велосипедов: {unique_bikes}")

Всего уникальных велосипедов: 700


# Найти пользователей потративших на поездки более 3 часов

In [34]:
users_over_3_hours = trips.filter(F.col("zip_code").isNotNull()) \
    .groupBy("zip_code") \
    .agg(F.sum("duration").alias("total_duration")) \
    .filter(F.col("total_duration") > 10800) \
    .orderBy(F.desc("total_duration"))

print(f"Найдено пользователей: {users_over_3_hours.count()}")
print("Результаты:")
users_over_3_hours.show(20)

Найдено пользователей: 3660
Результаты:
+--------+--------------+
|zip_code|total_duration|
+--------+--------------+
|   94107|      49757162|
|     nil|      45725550|
|   94105|      25596128|
|   94133|      21637675|
|   94102|      19128021|
|   94103|      19127388|
|   95531|      17270400|
|   94111|      14244997|
|   95112|      12742370|
|   94109|      12057128|
|   94040|       7807926|
|   94110|       7421936|
|   94117|       6901313|
|   94301|       6590378|
|   94041|       6276284|
|   94158|       6248167|
|   94306|       5550643|
|   94025|       5178237|
|   94108|       5127562|
|   94611|       5014906|
+--------+--------------+
only showing top 20 rows
